In [ ]:
import os, sys
import pickle

import matplotlib_inline
sys.path.append("../")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from matplotlib import gridspec
from matplotlib import ticker
from matplotlib.lines import Line2D

from tqdm import tqdm

from scipy.interpolate import interp1d
import numpy as np

matplotlib_inline.backend_inline.set_matplotlib_formats('retina')

%matplotlib inline
%load_ext autoreload
%autoreload 2
# Load plot settings
import healpy as hp
from plot_params import params
pylab.rcParams.update(params)

cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
output = "/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src_0.1mJy_new_freq/limits"

# mA_list = np.geomspace(8e-15, 2e-13, 5)
mA_list1 = np.geomspace(8e-15, 2e-13, 5)
mA_list2 = np.geomspace(13e-15, 1.3e-13, 4)
mA_list = np.geomspace(5e-15, 1e-13, 15)
# mA_list = np.sort(np.concatenate([mA_list1, mA_list2]))
eps_ary = np.geomspace(1e-9, 5e-6, 50_000)
from scipy.optimize import fsolve, minimize

def get_21cmfast_lims(mA, jobid):
    survey = "roman"
    obs_auto_rnom, obs_auto_xi = np.load(f"{output}/21cmfast_{survey}_pyilc_limits{jobid}_mA_{mA:.3e}_obs_auto_xi.npy")
    pred_auto_rnom, pred_auto_xi = np.load(f"{output}/21cmfast_{survey}_pyilc_limits{jobid}_mA_{mA:.3e}_pred_auto_xi.npy")
    obs_auto_cov = np.load(f"{output}/21cmfast_{survey}_pyilc_limits{jobid}_mA_{mA:.3e}_obs_auto_cov.npy")

    obs_cross_rnom, obs_cross_xi = np.load(f"{output}/21cmfast_{survey}_pyilc_limits{jobid}_mA_{mA:.3e}_obs_cross_xi.npy")
    pred_cross_rnom, pred_cross_xi = np.load(f"{output}/21cmfast_{survey}_pyilc_limits{jobid}_mA_{mA:.3e}_pred_cross_xi.npy")
    obs_cross_cov = np.load(f"{output}/21cmfast_{survey}_pyilc_limits{jobid}_mA_{mA:.3e}_obs_cross_cov.npy")

    @np.vectorize
    def auto_log_L(eps4: float):
        Log_l = (
            -0.5
            * (eps4 * pred_auto_xi - obs_auto_xi)
            @ np.linalg.inv(obs_auto_cov)
            @ (eps4 * pred_auto_xi - obs_auto_xi)
        )
        return Log_l

    @np.vectorize
    def cross_log_L(eps2: float):
        Log_l = (
            -0.5
            * (eps2 * pred_cross_xi - obs_cross_xi)
            @ np.linalg.inv(obs_cross_cov)
            @ (eps2 * pred_cross_xi - obs_cross_xi)
        )
        return Log_l
    max_auto_eps = minimize(lambda eps: -auto_log_L(eps**4), x0=0, options={"fatol":0.001}, method="Nelder-Mead").x
    max_cross_eps = minimize(lambda eps: -cross_log_L(eps**2), x0=1e-6,tol=1e-8, method="Nelder-Mead").x
    def auto_lam(eps4):
        max_log_L = auto_log_L(max_auto_eps**4)
        if eps4 < max_auto_eps**4:
            return 1
        else:
            return np.exp(auto_log_L(eps4) - max_log_L)

    def cross_lam(eps2):
        max_log_L = cross_log_L(max_cross_eps**2)
        if eps2 < max_cross_eps**2:
            return 1
        else:
            return np.exp(cross_log_L(eps2) - max_log_L)
    auto_lim = minimize(lambda eps: np.abs(2.71 + 2 * np.log(auto_lam(eps**4))), x0=max_auto_eps, options={"xatol":1e-10}, method="Nelder-Mead").x
    cross_lim = minimize(lambda eps: np.abs(2.71 + 2 * np.log(cross_lam(eps**2))), x0=max_auto_eps,  options={"xatol":1e-10}, method="Nelder-Mead").x
        # print(eps_ary[np.argmax(auto_log_L_eval)])
    # auto_log_L_eval[:np.argmax(auto_log_L_eval)] = 0
    # cross_log_L_eval = cross_log_L(eps_ary**2)
    # cross_log_L_eval[:np.argmax(cross_log_L_eval)] = 0
    # if jobid == 1:
    #     print(eps_ary[np.argmax(cross_log_L_eval)])
    # auto_lam = np.exp(auto_log_L_eval)/np.max(np.exp(auto_log_L_eval))
    # auto_lam[:np.argmax(auto_lam)] = 1
    # cross_lam = np.exp(cross_log_L_eval)/np.max(np.exp(cross_log_L_eval))
    # plt.loglog(eps_ary, -2 * np.log(cross_lam))
    # plt.axhline(2.71)
    # max_eps = minimize(lambda eps: -cross_log_L(eps**2), 1e-7)
    # # plt.ylim(0, 10)
    # cross_lam[:np.argmax(cross_lam)] = 1
    # auto_lim = eps_ary[np.argmin(np.abs(2.71 + 2 * np.log(auto_lam)))]
    # cross_lim = eps_ary[np.argmin(np.abs(2.71 + 2 * np.log(cross_lam)))]
    return cross_lim[0], auto_lim[0]#, auto_log_L, cross_log_L


In [ ]:
cross_results = []
for mA in tqdm(mA_list):
    for jobid in range(1, 26):
        cross_lim, auto_lim  = get_21cmfast_lims(mA, jobid)
        cross_results.append([mA, auto_lim, cross_lim])

cross_results = np.array(cross_results)


In [ ]:
fixed_cross_results = cross_results[:,1:].reshape(15, 25, 2)


In [ ]:
np.save(f"{output}/21cmfast_mA.npy", mA_list)
np.save(f"{output}/21cmfast_lims.npy", fixed_cross_results)

In [ ]:
analytic_theta = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/analytic_data/analytic_theta_ary.npy")
new_analytic_mA = np.sort(np.concatenate([np.geomspace(1e-15, 1e-14, 10), np.geomspace(5e-15, 1e-13, 15)[::2]]))
print(np.sort(new_analytic_mA))
def get_21cmfast_analytic_lims(mA, jobid):
    survey = "roman"
    analytic_w = np.load(f"/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/analytic_data/analytic_w_mA{mA:.3e}.npy")
    obs_auto_rnom, obs_auto_xi = np.load(f"{output}/21cmfast_{survey}_pyilc_limits{jobid}_mA_{mA:.3e}_obs_auto_xi.npy")
    pred_auto_rnom, pred_auto_xi = np.load(f"{output}/21cmfast_{survey}_pyilc_limits{jobid}_mA_{mA:.3e}_pred_auto_xi.npy")
    obs_auto_cov = np.load(f"{output}/21cmfast_{survey}_pyilc_limits{jobid}_mA_{mA:.3e}_obs_auto_cov.npy")
    total_xi = (2.73e3)**2 * interp1d(analytic_theta, analytic_w, bounds_error=False, fill_value=0)(pred_auto_rnom) + pred_auto_xi
    @np.vectorize
    def auto_log_L(eps4: float):
        Log_l = (
            -0.5
            * (eps4 * total_xi - obs_auto_xi)
            @ np.linalg.inv(obs_auto_cov)
            @ (eps4 * total_xi - obs_auto_xi)
        )
        return Log_l

    max_auto_eps = minimize(lambda eps: -auto_log_L(eps**4), x0=0, options={"fatol":0.001}, method="Nelder-Mead").x
    def auto_lam(eps4):
        max_log_L = auto_log_L(max_auto_eps**4)
        if eps4 < max_auto_eps**4:
            return 1
        else:
            return np.exp(auto_log_L(eps4) - max_log_L)

    auto_lim = minimize(lambda eps: np.abs(2.71 + 2 * np.log(auto_lam(eps**4))), x0=max_auto_eps, options={"xatol":1e-10}, method="Nelder-Mead").x

    return auto_lim[0], np.nan #, auto_log_L, cross_log_L

In [ ]:
new_analytic_results = []
for mA in tqdm(new_analytic_mA):
    for jobid in range(1, 26):
        auto_lim, cross_lim = get_21cmfast_analytic_lims(mA, jobid)
        new_analytic_results.append([mA, auto_lim, cross_lim])

new_analytic_results = np.array(new_analytic_results)
new_analytic_results = new_analytic_results[:,1:].reshape(18, 25, 2)


In [ ]:
np.save("/usr3/graduate/ebaker/dark_photon_constraints/notebooks_for_paper/data/21cmfast_analytic_mA.npy", new_analytic_mA)
np.save("/usr3/graduate/ebaker/dark_photon_constraints/notebooks_for_paper/data/21cmfast_analytic_lims.npy", new_analytic_results)

In [ ]:
plot_band(new_analytic_mA, new_analytic_results, auto=True, label="Analytic Auto", ls="--", color=cols_default[1])
plt.xscale("log")
plt.yscale("log")

In [ ]:
import pymaster as nmt
output = "/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src_0.1mJy_new_freq"
clean_auto_Cls = np.load(f"{output}/clean_auto_Cls_binning{150}.npy")
cross_Pg_Cls = np.load(f"{output}/cross_Pg_Cls_binning{150}_K22.npy")
# np.load(f"{output}/cross_Pg_Cls_binning{150}_K22.npy")
PPcovar = np.load(f"{output}/covariance_clean_PP_binning{150}_K22.npy")
    
gPcovar = np.load(f"{output}/covariance_clean_gP_binning{150}_K22.npy")
binning = nmt.NmtBin.from_nside_linear(2048, 150)
lrange = (binning.get_effective_ells() > 100) & (binning.get_effective_ells() < 4096)
eps_ary = np.geomspace(1e-9, 0.5e-7, 10_000)
def compute_halo_limits(mA):
    Tgamma0 = 2.73 * 1000 # mK
    pred_ls = np.geomspace(1, 7000, 50, dtype=int)
    pred_cross_Cls = -Tgamma0 * np.load(f"/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/halo_data/correct_virial_mass/Cl_gP_mA{mA:.3e}_modelK22.npy")
    pred_auto_Cls = Tgamma0**2 * np.load(f"/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/halo_data/correct_virial_mass/Cl_PP_mA{mA:.3e}.npy",)#/ (2*np.pi)**2

    interp_pred_cross_Cls = interp1d(pred_ls, pred_cross_Cls, fill_value=0) 
    interp_pred_auto_Cls = interp1d(pred_ls, pred_auto_Cls, fill_value=0)

    # only include ells above 200 where we can really be sure that the limber approximation is valid
    # lrange = (binning.get_effective_ells() > 200) & (binning.get_effective_ells() < 4000)
    lrange = (binning.get_effective_ells() > 150) & (binning.get_effective_ells() < 4096)

    
    @np.vectorize
    def cross_log_L(eps2):
        return -0.5 *(eps2 * interp_pred_cross_Cls(binning.get_effective_ells()[lrange]) - cross_Pg_Cls[lrange]) @ np.linalg.inv(gPcovar[lrange][:,lrange]) @ (eps2 * interp_pred_cross_Cls(binning.get_effective_ells()[lrange]) - cross_Pg_Cls[lrange])

    @np.vectorize
    def auto_log_L(eps4):
        return -0.5 *(eps4 * interp_pred_auto_Cls(binning.get_effective_ells()[lrange]) - clean_auto_Cls[lrange]) @ np.linalg.inv(PPcovar[lrange][:,lrange]) @ (eps4 * interp_pred_auto_Cls(binning.get_effective_ells()[lrange]) - clean_auto_Cls[lrange])

    max_auto_eps = minimize(lambda eps: -auto_log_L(eps**4), x0=0, options={"fatol":0.001}, method="Nelder-Mead").x
    max_cross_eps = minimize(lambda eps: -cross_log_L(eps**2), x0=0, options={"fatol":0.001}, method="Nelder-Mead").x
    def auto_lam(eps4):
        max_log_L = auto_log_L(max_auto_eps**4)
        if eps4 < max_auto_eps**4:
            return 1
        else:
            return np.exp(auto_log_L(eps4) - max_log_L)
    def cross_lam(eps2):
        max_log_L = cross_log_L(max_cross_eps**2)
        if eps2 < max_cross_eps**2:
            return 1
        else:
            return np.exp(cross_log_L(eps2) - max_log_L)
    # plt.loglog(eps_ary,-2 * np.log(np.vectorize(cross_lam)(eps_ary**2)))
    auto_lim = minimize(lambda eps: np.abs(2.71 + 2 * np.log(auto_lam(eps**4))), x0=max_auto_eps, options={"xatol":1e-12}, method="Nelder-Mead").x
    x0 = 1e-9
    x0_orig = x0
    counter = 0
    max_count = 20
    cross_lim = minimize(lambda eps: np.abs(2.71 + 2 * np.log(cross_lam(eps**2))), x0=x0,  options={"maxiter":10000, "fatol":1e-10, "xatol":1e-10}, method="Nelder-Mead").x
    while cross_lim == x0 and counter < max_count:
        x0 *= 2
        # print(cross_lim ,x0)
        cross_lim = minimize(lambda eps: np.abs(2.71 + 2 * np.log(cross_lam(eps**2))), x0=x0,  options={"maxiter":1000, "fatol":1e-10, "xatol":1e-1}, method="Nelder-Mead").x
        counter += 1
    if counter == max_count:
        print("Max iterations reached for cross lim")
        cross_lim = [1000.]
    # print(auto_lim, cross_lim)

    # print(f"Auto Limit is {auto_limit:.2e}. Cross Limit is {cross_limit:.2e} for mA = {mA:.3e}")
    return auto_lim[0], cross_lim[0], #cross_log_L, auto_log_L

mA_list = np.geomspace(1e-13, 1e-11, 25)

In [ ]:
compute_halo_limits(mA_list[16])

In [ ]:
# high_freq_01mJy_results = []
high_freq_01mJy_results = []

mA_list = np.geomspace(1e-13, 1e-11, 25)
# mA_list = [mA_list[10]]
print("covariance matrices made... computing limits")
for mA in tqdm(mA_list):
    auto_limit, cross_limit = compute_halo_limits(mA)
    high_freq_01mJy_results.append((mA, auto_limit, cross_limit))
high_freq_01mJy_results = np.array(high_freq_01mJy_results)
# np.save(f"{output}/local_halo_lims.npy", high_freq_01mJy_results)

In [ ]:
np.save(f"/usr3/graduate/ebaker/dark_photon_constraints/notebooks_for_paper/data/halo_mA.npy", mA_list)
np.save(f"/usr3/graduate/ebaker/dark_photon_constraints/notebooks_for_paper/data/halo_lims.npy", high_freq_01mJy_results)

In [ ]:
plt.plot(high_freq_01mJy_results[:,0], high_freq_01mJy_results[:,2])
plt.xscale("log")
plt.yscale("log")